# 18o — June 2026 HKO realised outcomes

This notebook extends the certified June 2026 Hong Kong Polymarket contract universe with official realised Hong Kong Observatory daily maximum temperatures.

It:

- requires the completed 18n June contract-event audit;
- retrieves and archives the official HKO daily maximum temperature data for station `HKO`;
- archives the official June 2026 Daily Extract and Monthly Weather Summary pages as source evidence;
- parses the HKO Open Data API response for all 30 June dates;
- attempts an independent parse of the Daily Extract table and compares it with the API values where possible;
- joins the 30 realised temperatures to all 330 certified contracts;
- applies the previously certified event-set boundaries without rounding the realised one-decimal HKO values;
- verifies exactly one winning contract per date;
- writes canonical outcome, integrity, issue, request-log, report and manifest outputs.

The notebook never creates a branch, commit, push, pull request, reminder or notification.

Official sources:

- HKO Daily Maximum Temperature Open Data API: `CLMMAXT`, station `HKO`;
- HKO Daily Extract for June 2026;
- HKO Monthly Weather Summary for June 2026.

In [1]:
from __future__ import annotations

import csv
import hashlib
import io
import json
import platform
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        "Run this notebook from the repository root. "
        f"Current directory: {REPO_ROOT}"
    )

STEP = "18o"
YEAR = 2026
MONTH = 6
EXPECTED_DATES = pd.date_range("2026-06-01", "2026-06-30", freq="D")

CONTRACT_AUDIT_PATH = (
    REPO_ROOT
    / "data/processed/18n_june_2026_contract_event_audit"
    / "18n_june_2026_contract_audit.csv"
)
EVENT_AUDIT_PATH = (
    REPO_ROOT
    / "data/processed/18n_june_2026_contract_event_audit"
    / "18n_june_2026_event_audit.csv"
)

RAW_DIR = REPO_ROOT / "data/raw/18o_june_2026_hko_realised_outcomes"
OUT_DIR = REPO_ROOT / "data/processed/18o_june_2026_hko_realised_outcomes"
REPORT_DIR = REPO_ROOT / "reports/18o_june_2026_hko_realised_outcomes"

for directory in (RAW_DIR, OUT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

API_CSV_URL = (
    "https://data.weather.gov.hk/weatherAPI/opendata/opendata.php"
    "?dataType=CLMMAXT&year=2026&month=6&rformat=csv&station=HKO"
)
API_JSON_URL = (
    "https://data.weather.gov.hk/weatherAPI/opendata/opendata.php"
    "?dataType=CLMMAXT&year=2026&month=6&rformat=json&station=HKO"
)
DAILY_EXTRACT_URL = (
    "https://www.hko.gov.hk/en/cis/dailyExtract.htm?m=06&y=2026"
)
MONTHLY_SUMMARY_URL = (
    "https://www.hko.gov.hk/en/wxinfo/pastwx/mws2026/mws202606.htm"
)

USER_AGENT = (
    "2026MScWeatherForecastingPolymarket/"
    "18o-june-2026-hko-realised-outcomes"
)
REQUEST_TIMEOUT_SECONDS = 90
MAX_ATTEMPTS = 4

session = requests.Session()
session.headers.update(
    {
        "User-Agent": USER_AGENT,
        "Accept": (
            "text/csv,application/json,text/html,"
            "application/xhtml+xml;q=0.9,*/*;q=0.8"
        ),
    }
)

if not CONTRACT_AUDIT_PATH.is_file():
    raise FileNotFoundError(
        "The certified 18n contract audit is missing: "
        f"{CONTRACT_AUDIT_PATH}"
    )

if not EVENT_AUDIT_PATH.is_file():
    raise FileNotFoundError(
        "The 18n date-level audit is missing: "
        f"{EVENT_AUDIT_PATH}"
    )

print(f"Repository root: {REPO_ROOT}")
print(f"Contract audit: {CONTRACT_AUDIT_PATH.relative_to(REPO_ROOT)}")
print(f"Event audit: {EVENT_AUDIT_PATH.relative_to(REPO_ROOT)}")

Repository root: /Users/edwardlee/Desktop/2026MScWeatherForecastingPolymarket
Contract audit: data/processed/18n_june_2026_contract_event_audit/18n_june_2026_contract_audit.csv
Event audit: data/processed/18n_june_2026_contract_event_audit/18n_june_2026_event_audit.csv


In [2]:
def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch_and_archive(
    *,
    source_name: str,
    url: str,
    raw_filename: str,
    accept: str | None = None,
) -> tuple[bytes, dict[str, Any]]:
    headers = {"Accept": accept} if accept else None
    last_error = ""

    for attempt in range(1, MAX_ATTEMPTS + 1):
        started = datetime.now(timezone.utc)
        try:
            response = session.get(
                url,
                headers=headers,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            metadata = {
                "source_name": source_name,
                "requested_url": url,
                "resolved_url": response.url,
                "retrieved_at_utc": started.isoformat(),
                "attempt": attempt,
                "http_status": response.status_code,
                "content_type": response.headers.get("Content-Type", ""),
                "content_length_header": response.headers.get(
                    "Content-Length", ""
                ),
                "error": "",
            }

            if response.status_code == 429 or response.status_code >= 500:
                last_error = f"HTTP {response.status_code}"
                time.sleep(1.5 * attempt)
                continue

            response.raise_for_status()
            payload = response.content
            if not payload:
                raise ValueError("Empty response body")

            raw_path = RAW_DIR / raw_filename
            raw_path.write_bytes(payload)

            metadata.update(
                {
                    "raw_path": str(raw_path.relative_to(REPO_ROOT)),
                    "size_bytes": len(payload),
                    "sha256": sha256_bytes(payload),
                }
            )
            return payload, metadata

        except (requests.RequestException, ValueError) as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            time.sleep(1.5 * attempt)

    raise RuntimeError(
        f"Failed to retrieve {source_name} after {MAX_ATTEMPTS} attempts: "
        f"{last_error}"
    )


def clean_numeric_text(value: Any) -> str:
    text = str(value).strip()
    text = text.replace("\ufeff", "")
    text = text.replace("°", "")
    text = re.sub(r"\s+", " ", text)
    return text


def parse_temperature(value: Any) -> float | None:
    if value is None:
        return None
    text = clean_numeric_text(value)
    if not text or text.lower() in {"nan", "na", "n/a", "---", "***"}:
        return None
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group(0)) if match else None


def parse_hko_csv(payload: bytes) -> pd.DataFrame:
    text = payload.decode("utf-8-sig", errors="replace")
    rows = list(csv.reader(io.StringIO(text)))

    parsed_rows: list[dict[str, Any]] = []

    # First try to locate a structured header.
    header_index = None
    header_normalised: list[str] = []
    for index, row in enumerate(rows):
        normalised = [
            re.sub(r"[^a-z0-9]+", " ", clean_numeric_text(cell).lower()).strip()
            for cell in row
        ]
        joined = " | ".join(normalised)
        if (
            "year" in normalised
            and "month" in normalised
            and "day" in normalised
            and (
                any("value" == cell for cell in normalised)
                or "temperature" in joined
                or "data value" in joined
            )
        ):
            header_index = index
            header_normalised = normalised
            break

    if header_index is not None:
        def find_index(candidates: list[str]) -> int | None:
            for candidate in candidates:
                for index, cell in enumerate(header_normalised):
                    if cell == candidate or candidate in cell:
                        return index
            return None

        year_index = find_index(["year"])
        month_index = find_index(["month"])
        day_index = find_index(["day"])
        value_index = find_index(
            [
                "value",
                "data value",
                "daily maximum temperature",
                "maximum temperature",
                "temperature",
            ]
        )
        completeness_index = find_index(
            ["data completeness", "completeness"]
        )

        if None not in (year_index, month_index, day_index, value_index):
            for row in rows[header_index + 1 :]:
                if len(row) <= max(
                    year_index,
                    month_index,
                    day_index,
                    value_index,
                ):
                    continue
                try:
                    year = int(float(clean_numeric_text(row[year_index])))
                    month = int(float(clean_numeric_text(row[month_index])))
                    day = int(float(clean_numeric_text(row[day_index])))
                except ValueError:
                    continue

                temperature = parse_temperature(row[value_index])
                if temperature is None:
                    continue

                parsed_rows.append(
                    {
                        "year": year,
                        "month": month,
                        "day": day,
                        "hko_daily_max_c": temperature,
                        "raw_value": clean_numeric_text(row[value_index]),
                        "data_completeness": (
                            clean_numeric_text(row[completeness_index])
                            if completeness_index is not None
                            and completeness_index < len(row)
                            else ""
                        ),
                        "parse_route": "structured_header",
                    }
                )

    # Fallback for HKO CSV variants with preamble and no easily recognised header.
    if not parsed_rows:
        for row in rows:
            if len(row) < 4:
                continue
            try:
                year = int(float(clean_numeric_text(row[0])))
                month = int(float(clean_numeric_text(row[1])))
                day = int(float(clean_numeric_text(row[2])))
            except ValueError:
                continue

            temperature = parse_temperature(row[3])
            if temperature is None:
                continue

            parsed_rows.append(
                {
                    "year": year,
                    "month": month,
                    "day": day,
                    "hko_daily_max_c": temperature,
                    "raw_value": clean_numeric_text(row[3]),
                    "data_completeness": (
                        clean_numeric_text(row[4]) if len(row) >= 5 else ""
                    ),
                    "parse_route": "positional_fallback",
                }
            )

    frame = pd.DataFrame(parsed_rows)
    if frame.empty:
        raise ValueError("No daily maximum rows could be parsed from HKO CSV")

    frame = frame.loc[
        frame["year"].eq(YEAR) & frame["month"].eq(MONTH)
    ].copy()
    frame["event_date"] = pd.to_datetime(
        {
            "year": frame["year"],
            "month": frame["month"],
            "day": frame["day"],
        }
    )

    return (
        frame[
            [
                "event_date",
                "hko_daily_max_c",
                "raw_value",
                "data_completeness",
                "parse_route",
            ]
        ]
        .drop_duplicates("event_date")
        .sort_values("event_date")
        .reset_index(drop=True)
    )

In [3]:
def normalise_json_field_name(value: Any) -> str:
    if isinstance(value, dict):
        value = (
            value.get("name")
            or value.get("fieldName")
            or value.get("id")
            or value.get("title")
            or json.dumps(value, ensure_ascii=False, sort_keys=True)
        )
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(value).lower(),
    ).strip()


def records_from_hko_json(payload: bytes) -> pd.DataFrame | None:
    try:
        parsed = json.loads(payload.decode("utf-8-sig", errors="strict"))
    except (UnicodeDecodeError, json.JSONDecodeError):
        return None

    candidates: list[pd.DataFrame] = []

    def visit(node: Any) -> None:
        if isinstance(node, dict):
            if isinstance(node.get("data"), list):
                data = node["data"]
                fields = node.get("fields")
                if data:
                    if isinstance(data[0], dict):
                        candidates.append(pd.DataFrame(data))
                    elif isinstance(data[0], (list, tuple)):
                        if isinstance(fields, list) and len(fields) == len(data[0]):
                            columns = [
                                normalise_json_field_name(field)
                                for field in fields
                            ]
                            candidates.append(pd.DataFrame(data, columns=columns))
                        else:
                            candidates.append(pd.DataFrame(data))
            for value in node.values():
                visit(value)
        elif isinstance(node, list):
            if node and isinstance(node[0], dict):
                candidates.append(pd.DataFrame(node))
            for value in node:
                visit(value)

    visit(parsed)

    for candidate in candidates:
        if candidate.empty:
            continue

        candidate = candidate.copy()
        candidate.columns = [
            normalise_json_field_name(column)
            for column in candidate.columns
        ]

        def find_column(terms: list[str]) -> str | None:
            for term in terms:
                for column in candidate.columns:
                    if column == term or term in column:
                        return column
            return None

        year_column = find_column(["year"])
        month_column = find_column(["month"])
        day_column = find_column(["day"])
        value_column = find_column(
            [
                "data value",
                "daily maximum temperature",
                "maximum temperature",
                "value",
                "temperature",
            ]
        )

        if None in (year_column, month_column, day_column, value_column):
            continue

        rows: list[dict[str, Any]] = []
        for _, row in candidate.iterrows():
            try:
                year = int(float(clean_numeric_text(row[year_column])))
                month = int(float(clean_numeric_text(row[month_column])))
                day = int(float(clean_numeric_text(row[day_column])))
            except (TypeError, ValueError):
                continue

            temperature = parse_temperature(row[value_column])
            if temperature is None:
                continue

            rows.append(
                {
                    "event_date": pd.Timestamp(year=year, month=month, day=day),
                    "hko_daily_max_c_json": temperature,
                }
            )

        frame = pd.DataFrame(rows)
        if frame.empty:
            continue

        frame = frame.loc[
            frame["event_date"].dt.year.eq(YEAR)
            & frame["event_date"].dt.month.eq(MONTH)
        ]
        if len(frame.drop_duplicates("event_date")) == 30:
            return (
                frame.drop_duplicates("event_date")
                .sort_values("event_date")
                .reset_index(drop=True)
            )

    return None


def flatten_column(column: Any) -> str:
    if isinstance(column, tuple):
        pieces = [str(piece) for piece in column if str(piece) != "nan"]
        column = " ".join(pieces)
    return re.sub(r"\s+", " ", str(column)).strip()


def parse_daily_extract_html(payload: bytes) -> pd.DataFrame | None:
    html = payload.decode("utf-8", errors="replace")
    try:
        tables = pd.read_html(io.StringIO(html))
    except ValueError:
        return None

    for table in tables:
        frame = table.copy()
        frame.columns = [flatten_column(column) for column in frame.columns]

        day_column = None
        max_column = None

        for column in frame.columns:
            normalised = re.sub(
                r"[^a-z0-9]+",
                " ",
                column.lower(),
            ).strip()
            if day_column is None and (
                normalised == "day"
                or normalised.startswith("day ")
                or normalised.endswith(" day")
                or "date" in normalised
            ):
                day_column = column

            if max_column is None and (
                "absolute daily max" in normalised
                or "absolute daily maximum" in normalised
                or (
                    "maximum temperature" in normalised
                    and "daily" in normalised
                )
            ):
                max_column = column

        if day_column is None or max_column is None:
            continue

        rows: list[dict[str, Any]] = []
        for _, row in frame.iterrows():
            day_match = re.search(r"\b([1-9]|[12]\d|3[01])\b", str(row[day_column]))
            temperature = parse_temperature(row[max_column])
            if day_match is None or temperature is None:
                continue

            day = int(day_match.group(1))
            rows.append(
                {
                    "event_date": pd.Timestamp(
                        year=YEAR,
                        month=MONTH,
                        day=day,
                    ),
                    "hko_daily_max_c_daily_extract": temperature,
                }
            )

        result = pd.DataFrame(rows)
        if len(result.drop_duplicates("event_date")) == 30:
            return (
                result.drop_duplicates("event_date")
                .sort_values("event_date")
                .reset_index(drop=True)
            )

    return None

In [4]:
contracts = pd.read_csv(CONTRACT_AUDIT_PATH, dtype={"yes_token_id": str, "no_token_id": str})
event_audit = pd.read_csv(EVENT_AUDIT_PATH)

contracts["event_date"] = pd.to_datetime(contracts["event_date"])
event_audit["event_date"] = pd.to_datetime(event_audit["event_date"])

required_contract_columns = {
    "event_date",
    "event_id",
    "event_slug",
    "market_id",
    "condition_id",
    "market_slug",
    "question",
    "event_type",
    "lower_bound_c",
    "upper_bound_c",
    "canonical_label",
    "yes_token_id",
    "no_token_id",
}
missing_contract_columns = required_contract_columns.difference(contracts.columns)
if missing_contract_columns:
    raise AssertionError(
        "18n contract audit is missing columns: "
        f"{sorted(missing_contract_columns)}"
    )

if len(event_audit) != 30:
    raise AssertionError(
        f"Expected 30 June event-audit rows, found {len(event_audit)}"
    )
if not event_audit["audit_status"].eq("PASS").all():
    raise AssertionError("At least one June event book is not certified PASS")
if len(contracts) != 330:
    raise AssertionError(
        f"Expected 330 certified June contracts, found {len(contracts)}"
    )
if contracts.duplicated(["event_date", "market_id"]).any():
    raise AssertionError("Duplicate date-market keys in the 18n contract audit")
if contracts["event_date"].nunique() != 30:
    raise AssertionError("The contract audit does not contain exactly 30 dates")

request_rows: list[dict[str, Any]] = []

csv_payload, csv_metadata = fetch_and_archive(
    source_name="HKO_CLMMAXT_CSV",
    url=API_CSV_URL,
    raw_filename="hko_clmmaxt_hko_2026_06.csv",
    accept="text/csv,*/*;q=0.8",
)
request_rows.append(csv_metadata)

json_payload, json_metadata = fetch_and_archive(
    source_name="HKO_CLMMAXT_JSON",
    url=API_JSON_URL,
    raw_filename="hko_clmmaxt_hko_2026_06.json",
    accept="application/json,*/*;q=0.8",
)
request_rows.append(json_metadata)

daily_extract_payload, daily_extract_metadata = fetch_and_archive(
    source_name="HKO_DAILY_EXTRACT_HTML",
    url=DAILY_EXTRACT_URL,
    raw_filename="hko_daily_extract_2026_06.html",
    accept="text/html,application/xhtml+xml;q=0.9,*/*;q=0.8",
)
request_rows.append(daily_extract_metadata)

monthly_summary_payload, monthly_summary_metadata = fetch_and_archive(
    source_name="HKO_MONTHLY_SUMMARY_HTML",
    url=MONTHLY_SUMMARY_URL,
    raw_filename="hko_monthly_summary_2026_06.html",
    accept="text/html,application/xhtml+xml;q=0.9,*/*;q=0.8",
)
request_rows.append(monthly_summary_metadata)

hko_daily = parse_hko_csv(csv_payload)
hko_json = records_from_hko_json(json_payload)
hko_daily_extract = parse_daily_extract_html(daily_extract_payload)

print(f"CSV daily rows parsed: {len(hko_daily)}")
print(
    "JSON cross-check rows parsed: "
    f"{0 if hko_json is None else len(hko_json)}"
)
print(
    "Daily Extract table rows parsed: "
    f"{0 if hko_daily_extract is None else len(hko_daily_extract)}"
)

CSV daily rows parsed: 30
JSON cross-check rows parsed: 30
Daily Extract table rows parsed: 0


In [5]:
issue_rows: list[dict[str, Any]] = []
check_rows: list[dict[str, Any]] = []

expected_frame = pd.DataFrame({"event_date": EXPECTED_DATES})
hko_daily = expected_frame.merge(hko_daily, on="event_date", how="left")

if hko_json is not None:
    hko_daily = hko_daily.merge(hko_json, on="event_date", how="left")
    hko_daily["api_json_match"] = np.isclose(
        hko_daily["hko_daily_max_c"],
        hko_daily["hko_daily_max_c_json"],
        atol=1e-12,
        equal_nan=False,
    )
else:
    hko_daily["hko_daily_max_c_json"] = np.nan
    hko_daily["api_json_match"] = pd.NA
    issue_rows.append(
        {
            "event_date": "",
            "market_id": "",
            "issue_level": "source_cross_check",
            "issue_code": "JSON_CROSS_CHECK_UNPARSEABLE",
            "detail": (
                "The official CSV was parsed successfully, but the optional "
                "JSON representation could not be independently parsed."
            ),
            "blocking": False,
        }
    )

if hko_daily_extract is not None:
    hko_daily = hko_daily.merge(
        hko_daily_extract,
        on="event_date",
        how="left",
    )
    hko_daily["daily_extract_match"] = np.isclose(
        hko_daily["hko_daily_max_c"],
        hko_daily["hko_daily_max_c_daily_extract"],
        atol=1e-12,
        equal_nan=False,
    )
else:
    hko_daily["hko_daily_max_c_daily_extract"] = np.nan
    hko_daily["daily_extract_match"] = pd.NA
    issue_rows.append(
        {
            "event_date": "",
            "market_id": "",
            "issue_level": "source_cross_check",
            "issue_code": "DAILY_EXTRACT_TABLE_UNPARSEABLE",
            "detail": (
                "The official Daily Extract page was archived, but its "
                "table was not machine-parseable in the downloaded HTML. "
                "Canonical values come from the official HKO CLMMAXT API."
            ),
            "blocking": False,
        }
    )

hko_daily["source_data_type"] = "CLMMAXT"
hko_daily["source_station"] = "HKO"
hko_daily["source_year"] = YEAR
hko_daily["source_month"] = MONTH
hko_daily["source_url"] = API_CSV_URL
hko_daily["daily_extract_url"] = DAILY_EXTRACT_URL
hko_daily["monthly_summary_url"] = MONTHLY_SUMMARY_URL
hko_daily["retrieved_at_utc"] = csv_metadata["retrieved_at_utc"]

missing_dates = hko_daily.loc[
    hko_daily["hko_daily_max_c"].isna(),
    "event_date",
]
for event_date in missing_dates:
    issue_rows.append(
        {
            "event_date": event_date.date().isoformat(),
            "market_id": "",
            "issue_level": "date",
            "issue_code": "HKO_DAILY_MAX_MISSING",
            "detail": "No official HKO daily maximum was parsed for this date.",
            "blocking": True,
        }
    )

if hko_daily["event_date"].duplicated().any():
    raise AssertionError("Duplicate dates in the canonical HKO outcome panel")

if hko_daily["hko_daily_max_c"].notna().sum() != 30:
    raise AssertionError(
        "The official HKO source did not provide all 30 June values"
    )

if not hko_daily["event_date"].equals(pd.Series(EXPECTED_DATES)):
    raise AssertionError("Canonical HKO outcome dates do not equal 1–30 June 2026")

if not hko_daily["hko_daily_max_c"].between(10.0, 45.0).all():
    raise AssertionError("At least one HKO maximum falls outside the integrity range")

# Preserve the official one-decimal values directly. Do not round to contract bins.
if not np.isclose(
    hko_daily["hko_daily_max_c"] * 10,
    np.round(hko_daily["hko_daily_max_c"] * 10),
    atol=1e-9,
).all():
    raise AssertionError("At least one HKO maximum is not recorded to 0.1°C")

if hko_json is not None and not hko_daily["api_json_match"].all():
    mismatches = hko_daily.loc[
        ~hko_daily["api_json_match"],
        [
            "event_date",
            "hko_daily_max_c",
            "hko_daily_max_c_json",
        ],
    ]
    raise AssertionError(
        "Official CSV and JSON values disagree:\n"
        + mismatches.to_string(index=False)
    )

if hko_daily_extract is not None and not hko_daily["daily_extract_match"].all():
    mismatches = hko_daily.loc[
        ~hko_daily["daily_extract_match"],
        [
            "event_date",
            "hko_daily_max_c",
            "hko_daily_max_c_daily_extract",
        ],
    ]
    raise AssertionError(
        "Official API and Daily Extract values disagree:\n"
        + mismatches.to_string(index=False)
    )

display(hko_daily)

,event_date,hko_daily_max_c,raw_value,data_completeness,parse_route,hko_daily_max_c_json,api_json_match,hko_daily_max_c_daily_extract,daily_extract_match,source_data_type,source_station,source_year,source_month,source_url,daily_extract_url,monthly_summary_url,retrieved_at_utc
0,2026-06-01,32.3,32.3,C,structured_header,32.3,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
1,2026-06-02,33.3,33.3,C,structured_header,33.3,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
2,2026-06-03,34.3,34.3,C,structured_header,34.3,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
3,2026-06-04,34.2,34.2,C,structured_header,34.2,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
4,2026-06-05,34.6,34.6,C,structured_header,34.6,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
5,2026-06-06,30.4,30.4,C,structured_header,30.4,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
6,2026-06-07,32.0,32.0,C,structured_header,32.0,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
7,2026-06-08,30.9,30.9,C,structured_header,30.9,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
8,2026-06-09,28.4,28.4,C,structured_header,28.4,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00
9,2026-06-10,28.6,28.6,C,structured_header,28.6,True,NaN,<NA>,CLMMAXT,HKO,2026,6,https://data.weather.gov.hk/weatherAPI/opendat...,https://www.hko.gov.hk/en/cis/dailyExtract.htm...,https://www.hko.gov.hk/en/wxinfo/pastwx/mws202...,2026-07-21T04:19:39.929112+00:00


In [6]:
outcome_panel = contracts.merge(
    hko_daily[
        [
            "event_date",
            "hko_daily_max_c",
            "raw_value",
            "data_completeness",
            "source_data_type",
            "source_station",
            "source_url",
            "daily_extract_url",
            "retrieved_at_utc",
        ]
    ],
    on="event_date",
    how="left",
    validate="many_to_one",
)

if outcome_panel["hko_daily_max_c"].isna().any():
    raise AssertionError("At least one certified contract has no HKO outcome")

def realised_yes(row: pd.Series) -> int:
    temperature = float(row["hko_daily_max_c"])
    event_type = row["event_type"]

    if event_type == "lower":
        return int(temperature < float(row["upper_bound_c"]))
    if event_type == "interior":
        return int(
            float(row["lower_bound_c"])
            <= temperature
            < float(row["upper_bound_c"])
        )
    if event_type == "upper":
        return int(temperature >= float(row["lower_bound_c"]))

    raise ValueError(f"Unknown event type: {event_type}")

outcome_panel["realised_yes"] = outcome_panel.apply(realised_yes, axis=1)
outcome_panel["realised_no"] = 1 - outcome_panel["realised_yes"]
outcome_panel["is_winning_contract"] = outcome_panel["realised_yes"].eq(1)

date_checks = (
    outcome_panel.groupby("event_date", as_index=False)
    .agg(
        hko_daily_max_c=("hko_daily_max_c", "first"),
        n_contracts=("market_id", "size"),
        n_unique_markets=("market_id", "nunique"),
        n_lower=("event_type", lambda values: int((values == "lower").sum())),
        n_interior=("event_type", lambda values: int((values == "interior").sum())),
        n_upper=("event_type", lambda values: int((values == "upper").sum())),
        n_yes=("realised_yes", "sum"),
        n_no=("realised_no", "sum"),
    )
)

winners = outcome_panel.loc[
    outcome_panel["realised_yes"].eq(1),
    [
        "event_date",
        "market_id",
        "market_slug",
        "canonical_label",
        "event_type",
        "lower_bound_c",
        "upper_bound_c",
        "hko_daily_max_c",
    ],
].rename(
    columns={
        "market_id": "winning_market_id",
        "market_slug": "winning_market_slug",
        "canonical_label": "winning_label",
        "event_type": "winning_event_type",
        "lower_bound_c": "winning_lower_bound_c",
        "upper_bound_c": "winning_upper_bound_c",
    }
)

date_checks = date_checks.merge(
    winners,
    on=["event_date", "hko_daily_max_c"],
    how="left",
    validate="one_to_one",
)

date_checks["exactly_one_winner"] = date_checks["n_yes"].eq(1)
date_checks["book_structure_valid"] = (
    date_checks["n_contracts"].eq(11)
    & date_checks["n_unique_markets"].eq(11)
    & date_checks["n_lower"].eq(1)
    & date_checks["n_interior"].eq(9)
    & date_checks["n_upper"].eq(1)
)
date_checks["date_pass"] = (
    date_checks["exactly_one_winner"]
    & date_checks["book_structure_valid"]
    & date_checks["winning_market_id"].notna()
)

if len(outcome_panel) != 330:
    raise AssertionError(
        f"Expected 330 contract outcomes, found {len(outcome_panel)}"
    )
if outcome_panel.duplicated(["event_date", "market_id"]).any():
    raise AssertionError("Duplicate date-market outcome keys")
if not outcome_panel["realised_yes"].isin([0, 1]).all():
    raise AssertionError("Non-binary realised_yes values found")
if int(outcome_panel["realised_yes"].sum()) != 30:
    raise AssertionError(
        f"Expected 30 winning contracts, found "
        f"{int(outcome_panel['realised_yes'].sum())}"
    )
if len(date_checks) != 30:
    raise AssertionError(
        f"Expected 30 date checks, found {len(date_checks)}"
    )
if not date_checks["date_pass"].all():
    bad = date_checks.loc[~date_checks["date_pass"]]
    raise AssertionError(
        "At least one June date failed outcome validation:\n"
        + bad.to_string(index=False)
    )

display(date_checks)

,event_date,hko_daily_max_c,n_contracts,n_unique_markets,n_lower,n_interior,n_upper,n_yes,n_no,winning_market_id,winning_market_slug,winning_label,winning_event_type,winning_lower_bound_c,winning_upper_bound_c,exactly_one_winner,book_structure_valid,date_pass
0,2026-06-01,32.3,11,11,1,9,1,1,10,2391366,highest-temperature-in-hong-kong-on-june-1-202...,32°C,interior,32.0,33.0,True,True,True
1,2026-06-02,33.3,11,11,1,9,1,1,10,2399845,highest-temperature-in-hong-kong-on-june-2-202...,33°C,interior,33.0,34.0,True,True,True
2,2026-06-03,34.3,11,11,1,9,1,1,10,2407124,highest-temperature-in-hong-kong-on-june-3-202...,34°C or higher,upper,34.0,NaN,True,True,True
3,2026-06-04,34.2,11,11,1,9,1,1,10,2416144,highest-temperature-in-hong-kong-on-june-4-202...,34°C or higher,upper,34.0,NaN,True,True,True
4,2026-06-05,34.6,11,11,1,9,1,1,10,2424599,highest-temperature-in-hong-kong-on-june-5-202...,34°C,interior,34.0,35.0,True,True,True
5,2026-06-06,30.4,11,11,1,9,1,1,10,2440923,highest-temperature-in-hong-kong-on-june-6-202...,30°C,interior,30.0,31.0,True,True,True
6,2026-06-07,32.0,11,11,1,9,1,1,10,2450209,highest-temperature-in-hong-kong-on-june-7-202...,32°C,interior,32.0,33.0,True,True,True
7,2026-06-08,30.9,11,11,1,9,1,1,10,2456236,highest-temperature-in-hong-kong-on-june-8-202...,30°C,interior,30.0,31.0,True,True,True
8,2026-06-09,28.4,11,11,1,9,1,1,10,2459844,highest-temperature-in-hong-kong-on-june-9-202...,28°C,interior,28.0,29.0,True,True,True
9,2026-06-10,28.6,11,11,1,9,1,1,10,2467883,highest-temperature-in-hong-kong-on-june-10-20...,28°C,interior,28.0,29.0,True,True,True


In [7]:
issue_columns = [
    "event_date",
    "market_id",
    "issue_level",
    "issue_code",
    "detail",
    "blocking",
]
issues = pd.DataFrame(issue_rows, columns=issue_columns)

blocking_issue_count = int(
    issues["blocking"].fillna(False).astype(bool).sum()
    if not issues.empty
    else 0
)
warning_count = int(len(issues) - blocking_issue_count)

if blocking_issue_count > 0:
    verdict = "NEEDS_CORRECTION"
elif warning_count > 0:
    verdict = "PASS_WITH_NONBLOCKING_SOURCE_WARNINGS"
else:
    verdict = "PASS"

hko_output_columns = [
    "event_date",
    "hko_daily_max_c",
    "raw_value",
    "data_completeness",
    "parse_route",
    "hko_daily_max_c_json",
    "api_json_match",
    "hko_daily_max_c_daily_extract",
    "daily_extract_match",
    "source_data_type",
    "source_station",
    "source_year",
    "source_month",
    "source_url",
    "daily_extract_url",
    "monthly_summary_url",
    "retrieved_at_utc",
]
hko_daily_output = hko_daily.reindex(columns=hko_output_columns).copy()
hko_daily_output["event_date"] = hko_daily_output["event_date"].dt.date.astype(str)

outcome_panel_output = outcome_panel.copy()
outcome_panel_output["event_date"] = (
    outcome_panel_output["event_date"].dt.date.astype(str)
)

date_checks_output = date_checks.copy()
date_checks_output["event_date"] = (
    date_checks_output["event_date"].dt.date.astype(str)
)

request_log = pd.DataFrame(request_rows)

hko_path = OUT_DIR / "18o_june_2026_hko_daily_max.csv"
outcome_path = OUT_DIR / "18o_june_2026_contract_outcomes.csv"
checks_path = OUT_DIR / "18o_june_2026_date_outcome_checks.csv"
winners_path = OUT_DIR / "18o_june_2026_winning_contracts.csv"
issues_path = OUT_DIR / "18o_june_2026_outcome_issues.csv"
request_log_path = OUT_DIR / "18o_june_2026_request_log.csv"

hko_daily_output.to_csv(hko_path, index=False)
outcome_panel_output.to_csv(outcome_path, index=False)
date_checks_output.to_csv(checks_path, index=False)
date_checks_output[
    [
        "event_date",
        "hko_daily_max_c",
        "winning_market_id",
        "winning_market_slug",
        "winning_label",
        "winning_event_type",
        "winning_lower_bound_c",
        "winning_upper_bound_c",
    ]
].to_csv(winners_path, index=False)
issues.to_csv(issues_path, index=False)
request_log.to_csv(request_log_path, index=False)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "verdict": verdict,
    "certified_june_dates_input": int(contracts["event_date"].nunique()),
    "certified_june_contracts_input": int(len(contracts)),
    "hko_daily_max_dates": int(hko_daily["hko_daily_max_c"].notna().sum()),
    "contract_outcome_rows": int(len(outcome_panel)),
    "winning_contracts": int(outcome_panel["realised_yes"].sum()),
    "dates_with_exactly_one_winner": int(
        date_checks["exactly_one_winner"].sum()
    ),
    "date_checks_passed": int(date_checks["date_pass"].sum()),
    "blocking_issue_rows": blocking_issue_count,
    "nonblocking_warning_rows": warning_count,
    "api_csv_json_cross_check_available": hko_json is not None,
    "api_csv_json_cross_check_passed": (
        bool(hko_daily["api_json_match"].all())
        if hko_json is not None
        else None
    ),
    "daily_extract_table_cross_check_available": (
        hko_daily_extract is not None
    ),
    "daily_extract_table_cross_check_passed": (
        bool(hko_daily["daily_extract_match"].all())
        if hko_daily_extract is not None
        else None
    ),
    "canonical_outcome_source": (
        "Hong Kong Observatory Open Data API, "
        "CLMMAXT, station HKO, June 2026"
    ),
    "settlement_semantics": (
        "Official HKO Daily Extract, Absolute Daily Maximum "
        "Temperature, one decimal place"
    ),
    "interpretation": (
        "All 30 certified June books have official HKO outcomes "
        "and exactly one winning contract."
    ),
}

summary_path = OUT_DIR / "18o_june_2026_outcome_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "requests": requests.__version__,
    "api_csv_url": API_CSV_URL,
    "api_json_url": API_JSON_URL,
    "daily_extract_url": DAILY_EXTRACT_URL,
    "monthly_summary_url": MONTHLY_SUMMARY_URL,
}
environment_path = OUT_DIR / "18o_june_2026_environment.json"
environment_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "step": "18o",
  "generated_at_utc": "2026-07-21T04:19:44.610497+00:00",
  "verdict": "PASS_WITH_NONBLOCKING_SOURCE_WARNINGS",
  "certified_june_dates_input": 30,
  "certified_june_contracts_input": 330,
  "hko_daily_max_dates": 30,
  "contract_outcome_rows": 330,
  "winning_contracts": 30,
  "dates_with_exactly_one_winner": 30,
  "date_checks_passed": 30,
  "blocking_issue_rows": 0,
  "nonblocking_warning_rows": 1,
  "api_csv_json_cross_check_available": true,
  "api_csv_json_cross_check_passed": true,
  "daily_extract_table_cross_check_available": false,
  "daily_extract_table_cross_check_passed": null,
  "canonical_outcome_source": "Hong Kong Observatory Open Data API, CLMMAXT, station HKO, June 2026",
  "settlement_semantics": "Official HKO Daily Extract, Absolute Daily Maximum Temperature, one decimal place",
  "interpretation": "All 30 certified June books have official HKO outcomes and exactly one winning contract."
}


In [8]:
report_lines = [
    "# 18o June 2026 HKO realised-outcome report",
    "",
    f"Generated at UTC: `{summary['generated_at_utc']}`",
    "",
    "## Overall judgement",
    "",
    f"**{verdict}**",
    "",
    "## Outcome construction",
    "",
    (
        "Official daily maximum temperatures were obtained from the "
        "Hong Kong Observatory Open Data API for `CLMMAXT`, station "
        "`HKO`, and June 2026. The official Daily Extract and Monthly "
        "Weather Summary pages were archived as source evidence."
    ),
    "",
    (
        "The one-decimal HKO value was used directly. No nearest-integer "
        "or nearest-bin rounding was applied."
    ),
    "",
    "## Sample flow",
    "",
    f"- Certified June dates received from 18n: {summary['certified_june_dates_input']}",
    f"- Certified June contracts received from 18n: {summary['certified_june_contracts_input']}",
    f"- Official HKO daily maxima obtained: {summary['hko_daily_max_dates']}",
    f"- Contract outcome rows created: {summary['contract_outcome_rows']}",
    f"- Winning contracts: {summary['winning_contracts']}",
    (
        "- Dates with exactly one winner: "
        f"{summary['dates_with_exactly_one_winner']}"
    ),
    f"- Date-level checks passed: {summary['date_checks_passed']}",
    f"- Blocking issue rows: {summary['blocking_issue_rows']}",
    (
        "- Non-blocking source warnings: "
        f"{summary['nonblocking_warning_rows']}"
    ),
    "",
    "## Source cross-checks",
    "",
    (
        "- CSV–JSON API cross-check available: "
        f"{summary['api_csv_json_cross_check_available']}"
    ),
    (
        "- CSV–JSON API cross-check passed: "
        f"{summary['api_csv_json_cross_check_passed']}"
    ),
    (
        "- Daily Extract table cross-check available: "
        f"{summary['daily_extract_table_cross_check_available']}"
    ),
    (
        "- Daily Extract table cross-check passed: "
        f"{summary['daily_extract_table_cross_check_passed']}"
    ),
    "",
    "## Winning contracts",
    "",
    (
        "| Date | HKO maximum | Winning label | Event type | "
        "Lower bound | Upper bound |"
    ),
    "|---|---:|---|---|---:|---:|",
]

for row in date_checks_output.itertuples(index=False):
    report_lines.append(
        "| {date} | {temp:.1f} | {label} | {event_type} | "
        "{lower} | {upper} |".format(
            date=row.event_date,
            temp=float(row.hko_daily_max_c),
            label=row.winning_label,
            event_type=row.winning_event_type,
            lower=(
                ""
                if pd.isna(row.winning_lower_bound_c)
                else f"{float(row.winning_lower_bound_c):.1f}"
            ),
            upper=(
                ""
                if pd.isna(row.winning_upper_bound_c)
                else f"{float(row.winning_upper_bound_c):.1f}"
            ),
        )
    )

report_lines.extend(
    [
        "",
        "## Event-set convention",
        "",
        "- Upper tail: `[K, infinity)`.",
        "- Interior label `k°C`: `[k, k+1)`.",
        "- Lower endpoint `k°C or below`: `(-infinity, k+1)`.",
        "",
        (
            "Only dates passing the outcome audit may proceed to June "
            "market-history and deterministic-weather reconstruction."
        ),
    ]
)

report_path = REPORT_DIR / "18o_june_2026_hko_realised_outcome_report.md"
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

# Build the manifest only after every canonical file is final.
manifest_rows: list[dict[str, Any]] = []
for root in (RAW_DIR, OUT_DIR, REPORT_DIR):
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18o_june_2026_sha256_manifest.csv":
            continue
        manifest_rows.append(
            {
                "path": str(path.relative_to(REPO_ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = OUT_DIR / "18o_june_2026_sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)

print(f"Report: {report_path.relative_to(REPO_ROOT)}")
print(f"Manifest entries: {len(manifest_rows)}")

Report: reports/18o_june_2026_hko_realised_outcomes/18o_june_2026_hko_realised_outcome_report.md
Manifest entries: 13


In [9]:
print("Date-level outcome checks:")
display(
    date_checks_output[
        [
            "event_date",
            "hko_daily_max_c",
            "winning_label",
            "winning_event_type",
            "n_yes",
            "exactly_one_winner",
            "date_pass",
        ]
    ]
)

print("Issue table:")
if issues.empty:
    print("No issue rows.")
else:
    display(issues)

print(f"Final verdict: {verdict}")

Date-level outcome checks:


,event_date,hko_daily_max_c,winning_label,winning_event_type,n_yes,exactly_one_winner,date_pass
0,2026-06-01,32.3,32°C,interior,1,True,True
1,2026-06-02,33.3,33°C,interior,1,True,True
2,2026-06-03,34.3,34°C or higher,upper,1,True,True
3,2026-06-04,34.2,34°C or higher,upper,1,True,True
4,2026-06-05,34.6,34°C,interior,1,True,True
5,2026-06-06,30.4,30°C,interior,1,True,True
6,2026-06-07,32.0,32°C,interior,1,True,True
7,2026-06-08,30.9,30°C,interior,1,True,True
8,2026-06-09,28.4,28°C,interior,1,True,True
9,2026-06-10,28.6,28°C,interior,1,True,True


Issue table:


,event_date,market_id,issue_level,issue_code,detail,blocking
0,,,source_cross_check,DAILY_EXTRACT_TABLE_UNPARSEABLE,"The official Daily Extract page was archived, ...",False


Final verdict: PASS_WITH_NONBLOCKING_SOURCE_WARNINGS
